In [12]:
import jax
from flax import nnx
import jax.numpy as jnp
import optax

In [ ]:
# model simple vector field network v_theta(x, t)
class HiddenNet(nnx.Module):
    def __init__(self, dim, *, rngs: nnx.Rngs) -> None:
        self.linear = nnx.Linear(dim, dim, rngs=rngs)

    def __call__(self, x):
        return nnx.softplus(self.linear(x))

class VectorFieldNetwork(nnx.Module):

    def __init__(self, input_dim=2, hidden_dim=64, num_hidden_layers=5, *, rngs: nnx.Rngs) -> None:
        self.linear1 = nnx.Linear(input_dim + 1, hidden_dim, rngs=rngs)

        @nnx.split_rngs(splits=num_hidden_layers)
        @nnx.vmap(in_axes=0, out_axes=0)
        def create_hidden_layer(rngs):
            return HiddenNet(hidden_dim, rngs=rngs)

        self.hidden_layers = create_hidden_layer(rngs)
        self.linear3 = nnx.Linear(hidden_dim, input_dim, rngs=rngs)

    def __call__(self, x, t):
        # x [batch_size, input_dim]
        # t [batch_size, 1]
        inputs = jnp.concatenate([x, t], axis=-1) # [batch_size, input_dim + 1]
        out1 = nnx.softplus(self.linear1(inputs))

        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def apply_hidden_layers(x, layer):
            x = layer(x)
            return x

        h = apply_hidden_layers(out1, self.hidden_layers)
        return self.linear3(h) # [batch_size, input_dim]

In [101]:
# 2. Helper to generate toy 2D target data (a circle)
def sample_target_data(batch_size, rng_key):
    k1, k2 = jax.random.split(rng_key)
    theta = jax.random.uniform(k1, (batch_size, 1)) * 2 * jnp.pi
    r = 2.0 + jax.random.uniform(k2, (batch_size, 1)) * 0.1
    x1 = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=-1)  # [batch_size, 2]
    return x1

print(sample_target_data(5, jax.random.key(0)).shape)


(5, 2)


In [ ]:
@nnx.jit
def train_step(model, optimizer, xt, t, u_t):
    def loss_fn(model):
        pred = model(xt, t)  # [batch_size, 2]
        return jnp.mean((pred - u_t) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss



def train_flow_matching(epochs=1000, batch_size=128, learning_rate=1e-3):
    model = VectorFieldNetwork(input_dim=2, hidden_dim=128, rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)

    print("Starting training flow matching...")

    for epoch in range(epochs):
        epoch_key = jax.random.key(epoch)
        k1, k2, k3 = jax.random.split(epoch_key, num=3)

        # sample endpoint data: x1 ~ target distribution, x0 ~ simple prior (e.g., Gaussian)
        x1 = sample_target_data(batch_size, k1)
        x0 = jax.random.normal(k2, (batch_size, 2))  # [batch_size, 2]

        # sample random time t ~ Uniform(0, 1)
        t = jax.random.uniform(k3, (batch_size, 1)) # [batch_size, 1]

        # Construct the conditional path (Linear interpolation / Optimal Transport)
        xt = (1 - t) * x0 + t * x1  # [batch_size, 2]

        # Target velocity field u_t(x | x_0, x_1) = x_1 - x_0
        u_t = x1 - x0  # [batch_size, 2]

        loss = train_step(model, optimizer, xt, t, u_t)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return model


In [ ]:
# Sampling / Inference Function (Solving the ODE via Euler Integration)
@jax.jit(static_argnames=["num_samples", "num_steps"])
def sample_from_model(model, num_samples=1000, num_steps=50):
    model.eval()
    key = jax.random.key(42)
    # sample initial pure noise from simple prior x_0 ~ N(0, I)
    xt = jax.random.normal(key, (num_samples, 2))
    dt = 1.0 / num_steps

    def euler_step(i, carry):
        xt, model = carry
        t_val = i * dt
        t = jnp.full((num_samples, 1), t_val) # [num_samples, 1]

        vt = model(xt, t)  # [num_samples, 2]
        xt = xt + vt * dt
        return (xt , model)

    xt, _ = jax.lax.fori_loop(jnp.int32(0), num_steps,euler_step , (xt, model))

    return xt


In [124]:
trained_model = train_flow_matching(epochs=1001, batch_size=256, learning_rate=1e-3)


None
Starting training flow matching...
Epoch 0, Loss: 3.8360
Epoch 100, Loss: 2.8764
Epoch 200, Loss: 2.2474
Epoch 300, Loss: 2.1587
Epoch 400, Loss: 1.8608
Epoch 500, Loss: 2.2304
Epoch 600, Loss: 2.1449
Epoch 700, Loss: 2.0798
Epoch 800, Loss: 1.6796
Epoch 900, Loss: 1.7540
Epoch 1000, Loss: 2.1063


In [129]:

# Sample from the trained model
generated_samples = sample_from_model(trained_model, num_samples=5, num_steps=50)
print("\nGenerated 2D coordinates on the learned circle:")
print(generated_samples)


Generated 2D coordinates on the learned circle:
[[ 0.26678634  1.9040599 ]
 [ 1.839187    0.46235162]
 [-0.8044956   1.7033975 ]
 [-1.8059622   1.0032756 ]
 [ 1.330911    1.4711717 ]]
